In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
PARQ_PATH = Path('../sample_dataset/processed_data/parquets/20220730-0002.parquet')
DC_OFFSET = 1.9  # Volts

In [ ]:
# Load the parquet file into a DataFrame
df = pd.read_parquet(PARQ_PATH)
df.columns = df.columns.astype('int16') # parquet stores column names as strings; we want integers

num_initial_signals = df.shape[0]
print(f"\x1b[1;36m{num_initial_signals}\x1b[0m signals loaded.")

In [ ]:
# Drop any signals that have missing or infinite values
df = df.replace([np.inf, -np.inf], np.nan).dropna(how='any')
num_missing_signals = num_initial_signals - df.shape[0]
print(f"Removed \x1b[1;31m{num_missing_signals}\x1b[0m signals with missing or infinite values.")

In [ ]:
# Subtract DC offset
df = df - DC_OFFSET
# Invert signals
df = -1 * df

In [ ]:
# Offset dataframe upwards by global minimum
# TODO: check if this is necessary
df = df + abs(min(df.min()))

In [ ]:
# Plot signals
fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df.T
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

fig.show()

In [ ]:
# Remove signals with initial values greater than some threshold
threshold = 0.15  # volts
filt = (df[0] < threshold)
df_clean = df[filt]

num_filtered_signals = df.shape[0] - df_clean.shape[0]

print(
    f"Removed \x1b[1;31m{num_filtered_signals}\x1b[0m signals "
    f"with starting values greater than \x1b[1;31m{threshold}\x1b[0m volts."
)

In [ ]:
# Plot filtered signals
fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df_clean.T
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

fig.show()